### ЗАДАЧА: Архитектура компенсаций (ABC + наследование + композиция)

В компании есть сотрудники разных ролей: разработчики, менеджеры и аналитики.
Нужно собрать ООП-модель расчёта выплат и отчёт по отделам.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Абстрактный класс `EmployeeBase`:
   - поля `name`, `base_salary`, `department`
   - абстрактный метод `total_pay()`
   - метод `short()` для короткого описания сотрудника.

2. Классы-наследники:
   - `Developer(name, base_salary, department, overtime_hours, overtime_rate)`
   - `Manager(name, base_salary, department, bonus)`
   - `Analyst(name, base_salary, department, reports_done, report_rate)`
   - у каждого свой расчёт `total_pay()`.

3. Класс `DepartmentBudget` (композиция):
   - хранит `department_name` и список сотрудников отдела
   - методы `add_employee(emp)`, `department_total()`, `top_paid()`.

4. Класс `PayrollService`:
   - принимает список сотрудников
   - метод `total_company_payroll()`
   - метод `totals_by_department()` -> словарь `{department: сумма}`
   - метод `highest_paid_employee()`.

5. Добавить dunder-методы:
   - `__repr__` для сотрудников
   - `__lt__` для сравнения сотрудников по `total_pay()`.

6. На основе входных данных создать объекты и вывести:
   - общий payroll,
   - выплаты по отделам,
   - самого дорогого сотрудника.

ПОДСКАЗКИ ПО РАСЧЁТАМ:
- `Developer.total_pay()` -> `base_salary + overtime_hours * overtime_rate`.
- `Manager.total_pay()` -> `base_salary + bonus`.
- `Analyst.total_pay()` -> `base_salary + reports_done * report_rate`.
- В `totals_by_department()` удобно использовать `dict.get(dep, 0)`.
- В `top_paid()` и `highest_paid_employee()` использовать `max(..., key=...)`.


In [ ]:
from abc import ABC, abstractmethod

staff_rows = [
    ("Developer", "Алиса", 2200, "Engineering", 12, 25),
    ("Manager", "Боб", 2800, "Engineering", 900),
    ("Analyst", "Кира", 2000, "Data", 14, 40),
    ("Developer", "Данил", 2400, "Engineering", 4, 25),
    ("Manager", "Ева", 2600, "Data", 600),
    ("Analyst", "Жан", 1900, "Operations", 20, 30),
]


class EmployeeBase(ABC):
    # TODO: __init__(name, base_salary, department)
    # TODO: short() -> строка "Имя (Department)"
    def __init__(self, name, base_salary, department):
        self.name = name
        self.base_salary = base_salary
        self.department = department

    def short(self):
        return f"{self.name} ({self.department})"
    
    @abstractmethod
    def total_pay(self):
        pass

    # TODO: __repr__
    # TODO: __lt__(other) -> сравнение по total_pay()

    def __repr__(self):
        return f"{self.__class__.__name__}({self.name}, {self.base_salary}, {self.department})"

    def __lt__(self, other):
        return self.total_pay() < other.total_pay()

class Developer(EmployeeBase):
    # TODO: __init__(name, base_salary, department, overtime_hours, overtime_rate)
    # TODO: total_pay()
    
    def __init__(self, name, base_salary, department, overtime_hours, overtime_rate):
        super().__init__(name, base_salary, department)
        self.overtime_hours = overtime_hours
        self.overtime_rate = overtime_rate

    def total_pay(self):
        return self.base_salary + self.overtime_hours * self.overtime_rate


class Manager(EmployeeBase):
    # TODO: __init__(name, base_salary, department, bonus)
    # TODO: total_pay()

    def __init__(self, name, base_salary, department, bonus):
        super().__init__(name, base_salary, department)
        self.bonus = bonus

    def total_pay(self):
        return self.base_salary + self.bonus

class Analyst(EmployeeBase):
    # TODO: __init__(name, base_salary, department, reports_done, report_rate)
    # TODO: total_pay()

    def __init__(self, name, base_salary, department, reports_done, report_rate):
        super().__init__(name, base_salary, department)
        self.reports_done = reports_done
        self.report_rate = report_rate

    def total_pay(self):
        return self.base_salary + self.reports_done * self.report_rate


class DepartmentBudget:
    # TODO: __init__(department_name)
    # TODO: add_employee(emp)
    # TODO: department_total()
    # TODO: top_paid()

    def __init__(self, department_name):
        self.department_name = department_name
        self.employees = []

    def add_employee(self, emp):
        if emp.department == self.department_name:
            self.employees.append(emp)

    def department_total(self):
        return sum(emp.total_pay() for emp in self.employees)

    def top_paid(self):
        if not self.employees:
            return None
        return max(self.employees, key=lambda emp: emp.total_pay())

class PayrollService:
    # TODO: __init__(employees)
    # TODO: total_company_payroll()
    # TODO: totals_by_department()
    # TODO: highest_paid_employee()

    def __init__(self, employees):
        self.employees = employees
        self.departments = {}
        for emp in employees:
            if emp.department not in self.departments:
                self.departments[emp.department] = DepartmentBudget(emp.department)
            self.departments[emp.department].add_employee(emp)

    def total_company_payroll(self):
        return sum(emp.total_pay() for emp in self.employees)

    def totals_by_department(self):
        return {dept: budget.department_total() for dept, budget in self.departments.items()}

    def highest_paid_employee(self):
        if not self.employees:
            return None
        return max(self.employees, key=lambda emp: emp.total_pay())


# TODO: создать объекты сотрудников из staff_rows
# TODO: создать PayrollService и вывести общий payroll
# TODO: вывести суммы по отделам
# TODO: вывести самого высокооплачиваемого сотрудника

employees = []
for row in staff_rows:
    position = row[0]
    name = row[1]
    base_salary = row[2]
    department = row[3]
    
    if position == "Developer":
        overtime_hours = row[4]
        overtime_rate = row[5]
        employee = Developer(name, base_salary, department, overtime_hours, overtime_rate)
    elif position == "Manager":
        bonus = row[4]
        employee = Manager(name, base_salary, department, bonus)
    elif position == "Analyst":
        reports_done = row[4]
        report_rate = row[5]
        employee = Analyst(name, base_salary, department, reports_done, report_rate)
    else:
        continue  # На случай, если появится неизвестная должность

    employees.append(employee)

# Создаём PayrollService и выводим общий payroll
payroll_service = PayrollService(employees)
print(f"Общая зарплата компании: {payroll_service.total_company_payroll()}")

# Выводим суммы по отделам
print("Суммы по отделам:")
for department, total in payroll_service.totals_by_department().items():
    print(f"{department}: {total}")

# Выводим самого высокооплачиваемого сотрудника
highest_paid = payroll_service.highest_paid_employee()
if highest_paid:
    print(f"Самый высокооплачиваемый сотрудник: {highest_paid.short()}, зарплата: {highest_paid.total_pay()}")
else:
    print("Сотрудники не найдены")

Общая зарплата компании: 16960
Суммы по отделам:
Engineering: 8700
Data: 5760
Operations: 2500
Самый высокооплачиваемый сотрудник: Боб (Engineering), зарплата: 3700
